In [1]:
# News Topic Public Opinion Mining Agent

### AI-Powered Public Opinion Analysis from YouTube Comments

An AI-powered NLP system that takes a news topic as input,
collects relevant YouTube comments, and analyzes public opinion
using:

- Named Entity Recognition
- Emotion Detection
- Stance Detection
- Aspect Extraction
- Comment Clustering
- AI-based Summarization
- Analytics Visualization
- Gemini-generated infographic

SyntaxError: invalid syntax (1599044245.py, line 5)

In [ ]:
!pip install -q google-api-python-client langdetect \
                ipywidgets matplotlib nrclex psutil \
                google-generativeai Pillow
!python -m spacy download en_core_web_sm

In [ ]:
# ============================================================
#   AI AGENT — DEEP OPINION MINING + GEMINI IMAGE GENERATION
#   NIT Warangal | M.Tech Minor Project
#   Emotion + Stance + Aspect + Gemini AI Image
# ============================================================
#  CELL 1 — run once, then restart runtime:
#
#  !pip install -q google-api-python-client langdetect \
#                  ipywidgets matplotlib nrclex psutil \
#                  google-generativeai Pillow
#  !python -m spacy download en_core_web_sm
# ============================================================

import warnings
warnings.filterwarnings("ignore")
import os, re, time, gc, random, io
from collections import Counter
from IPython.display import display, clear_output, HTML, Image as IPImage
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import psutil
from PIL import Image

import spacy
from langdetect import detect
from googleapiclient.discovery import build
from nrclex import NRCLex
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# Gemini imports
import google.generativeai as genai
from google import genai as genai2
from google.genai import types

# ── CONFIG ────────────────────────────────────────────────────
from google.colab import userdata

YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")   # get free from aistudio.google.com

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_VIDEOS           = 5
MAX_RAW_COMMENTS     = 1500
MAX_SAMPLE           = 800
MAX_EMOTION_COMMENTS = 300
MAX_ENTITY_COMMENTS  = 100
MIN_VIEWS            = 50000
N_CLUSTERS           = 8

# ── EMOTION PALETTE ───────────────────────────────────────────
EMO_COLORS = {
    "joy":      "#2DC653",
    "anger":    "#E63946",
    "sadness":  "#0E6BA8",
    "fear":     "#8338EC",
    "disgust":  "#FB8500",
    "surprise": "#06BEE1",
    "neutral":  "#64748B",
}
ALL_EMOTIONS = list(EMO_COLORS.keys())

STANCE_COLORS = {
    "Support":  "#166534",
    "Oppose":   "#991B1B",
    "Neutral":  "#64748B",
}

NRC_MAP = {
    "joy": "joy", "anger": "anger", "sadness": "sadness",
    "fear": "fear", "disgust": "disgust", "surprise": "surprise",
    "anticipation": "neutral", "trust": "neutral",
    "positive": "joy", "negative": "anger",
}

def safe(t): return re.sub(r"[^\w]", "_", t)
def mem():
    r = psutil.virtual_memory()
    print(f"  RAM: {r.used/1e9:.1f}GB / {r.total/1e9:.1f}GB")


# ════════════════════════════════════════════════════════════
# SECTION 1 — LOAD
# ════════════════════════════════════════════════════════════

print("━" * 55)
print("  AI Agent — Opinion Mining + Gemini Image")
print("  No torch / No transformers / No GPU")
print("━" * 55)

print("  ⟳ spaCy NER...")
nlp = spacy.load("en_core_web_sm")
nlp.select_pipes(enable=["ner"])

print("  ⟳ YouTube API...")
yt = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

print("  ⟳ Gemini API (text summaries)...")
genai.configure(api_key=GEMINI_API_KEY)
gemini_text = genai.GenerativeModel("gemini-1.5-flash-8b")

print("  ⟳ Gemini Image client...")
image_client = genai2.Client(api_key=GEMINI_API_KEY)

mem()
print("  ✅ Ready!\n")


# ════════════════════════════════════════════════════════════
# SECTION 2 — GEMINI TEXT SUMMARIES
# ════════════════════════════════════════════════════════════

def generate_analytics_image(topic, entities, overall_emos,
                              overall_stance_pct, entity_emos):
    top_emos    = overall_emos.most_common(3)
    emo_str     = ", ".join([f"{e}: {c} comments" for e, c in top_emos])
    entity_str  = ", ".join([e[0] for e in entities[:5]])
    sup         = overall_stance_pct.get("Support", 0)
    opp         = overall_stance_pct.get("Oppose",  0)
    neu         = overall_stance_pct.get("Neutral", 0)
    top_entity  = entities[0][0] if entities else "Public"
    top_entity_emo = ""
    if top_entity in entity_emos:
        te = entity_emos[top_entity]["emotion_counts"].most_common(1)
        if te: top_entity_emo = te[0][0]

    prompt = (
        f"Create a professional social media analytics infographic "
        f"for the news topic: '{topic}'. "
        f"Design a dark navy blue dashboard with golden title accents. "
        f"Include: "
        f"1. Bold title: 'Public Opinion Analysis — {topic}' "
        f"2. Key entities discussed: {entity_str} "
        f"3. Top emotions: {emo_str} shown as coloured bars "
        f"   (joy=green, anger=red, sadness=blue, fear=purple, neutral=grey) "
        f"4. Stance: {sup}% Support (green), {opp}% Oppose (red), "
        f"   {neu}% Neutral (grey) "
        f"5. Highlight box: '{top_entity}' feels '{top_entity_emo}' "
        f"6. Footer: 'AI Agent Analysis | NIT Warangal' "
        f"Modern infographic style, no real faces or people."
    )

    print("    Generating analytics image with Gemini...")

    try:
        response = image_client.models.generate_content(
            model    = "models/gemini-2.5-flash-image",
            contents = prompt,
            config   = types.GenerateContentConfig(
                response_modalities = ["TEXT", "IMAGE"]
            )
        )

        # extract image from response
        for part in response.candidates[0].content.parts:
            if part.inline_data is not None:
                img_data = part.inline_data.data
                img      = Image.open(io.BytesIO(img_data))
                path     = f"{OUTPUT_DIR}/{safe(topic)}_analytics_image.png"
                img.save(path)
                display(IPImage(path, width=800))
                print(f"    ✅ Image saved → {path}")
                return path

        # if no image part found
        print("    ⚠ Gemini returned no image — check model availability")
        return None

    except Exception as e:
        print(f"    ⚠ Image generation failed: {e}")
        print("    (Continuing without image)")
        return None


# ════════════════════════════════════════════════════════════
# SECTION 4 — YOUTUBE
# ════════════════════════════════════════════════════════════

def search_videos(query, max_results=15):
    try:
        res = yt.search().list(
            q=query, part="snippet", type="video",
            maxResults=max_results, order="relevance",
            relevanceLanguage="en",
        ).execute()
        return [i["id"]["videoId"] for i in res.get("items", [])]
    except:
        return []

def video_details(vid):
    try:
        res  = yt.videos().list(part="snippet,statistics", id=vid).execute()
        if not res["items"]: return None
        item = res["items"][0]
        return (
            item["snippet"]["title"],
            item["snippet"]["channelTitle"],
            int(item["statistics"].get("viewCount", 0)),
            int(item["statistics"].get("commentCount", 0)),
        )
    except:
        return None

def get_comments_fast(vid, max_comments=300):
    comments, nxt = [], None
    while len(comments) < max_comments:
        try:
            res = yt.commentThreads().list(
                part="snippet", videoId=vid,
                maxResults=min(100, max_comments - len(comments)),
                pageToken=nxt, textFormat="plainText",
                order="relevance",
            ).execute()
        except:
            break
        for i in res.get("items", []):
            comments.append(
                i["snippet"]["topLevelComment"]["snippet"]["textDisplay"])
        nxt = res.get("nextPageToken")
        if not nxt: break
    return comments


# ════════════════════════════════════════════════════════════
# SECTION 5 — PREPROCESSING
# ════════════════════════════════════════════════════════════

HINDI_NOISE = {
    "hai","hain","nhi","nahi","bhai","yaar","agar","aur","ke","ki",
    "ka","ko","se","me","mein","par","pe","liye","karegi","karega",
    "karna","karo","bahut","bohot","accha","acha","theek","sahi",
    "abhi","kab","kyun","kya","koi","kuch","sirf","toh","woh","yeh",
    "ye","jo","jab","log","desh","sara","puri","pura","ji","bhi",
    "hoga","tha","the","thi",
}
HINGLISH_MARKERS = {
    "hai","hain","nahi","nhi","bahut","bohot","bhai","yaar","karo",
    "acha","accha","theek","abhi","kyun","kya","aur","lekin","magar",
    "iska","uska","woh","yeh",
}

def is_valid_entity(text):
    t = text.strip().lower()
    if len(t) < 3: return False
    if t in HINDI_NOISE: return False
    if not text[0].isupper(): return False
    if not any(v in t for v in "aeiou"): return False
    c = re.sub(r"[^a-zA-Z ]", "", text)
    if len(c) < len(text) * 0.8: return False
    return True

def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def filter_comments(comments):
    out = []
    for c in comments:
        try:
            if detect(c) != "en": continue
            words = c.lower().split()
            if len(words) <= 4: continue
            hindi_ratio = sum(1 for w in words
                              if w in HINGLISH_MARKERS) / len(words)
            if hindi_ratio > 0.15: continue
            out.append(c)
        except:
            pass
    return out


# ════════════════════════════════════════════════════════════
# SECTION 6 — NER
# ════════════════════════════════════════════════════════════

def extract_entities(comments):
    ents = []
    for doc in nlp.pipe(comments[:500], batch_size=50):
        for e in doc.ents:
            if e.label_ in ["PERSON", "ORG", "GPE"]:
                name = e.text.strip().title()
                if is_valid_entity(name):
                    ents.append(name)
        del doc
    gc.collect()
    return Counter(ents).most_common(8)


# ════════════════════════════════════════════════════════════
# SECTION 7 — CLUSTERING
# ════════════════════════════════════════════════════════════

def cluster_comments(comments, n_clusters=None, label="overall"):
    if len(comments) < 15:
        return {0: comments}
    n = n_clusters or min(N_CLUSTERS, len(comments) // 8, 12)
    if n < 2: return {0: comments}
    try:
        vec  = TfidfVectorizer(max_features=300, stop_words="english",
                               min_df=2, ngram_range=(1, 2))
        X    = vec.fit_transform(comments)
        lbls = KMeans(n_clusters=n, random_state=42,
                      n_init=5, max_iter=100).fit_predict(X)
        del X, vec; gc.collect()
        clusters = {}
        for lbl, c in zip(lbls, comments):
            clusters.setdefault(int(lbl), []).append(c)
        print(f"    [{label}] {n} clusters from {len(comments)} comments")
        return clusters
    except Exception:
        return {0: comments}


# ════════════════════════════════════════════════════════════
# SECTION 8 — EMOTION DETECTION
# ════════════════════════════════════════════════════════════

def detect_emotions(comments, max_c=300):
    if not comments: return Counter()
    counter = Counter()
    for c in comments[:max_c]:
        try:
            nrc   = NRCLex(c)
            freqs = nrc.affect_frequencies
            specific = {k: v for k, v in freqs.items()
                        if k not in ("positive","negative") and v > 0}
            if specific:
                top    = max(specific, key=specific.get)
                mapped = NRC_MAP.get(top, "neutral")
            else:
                pos = freqs.get("positive", 0)
                neg = freqs.get("negative", 0)
                if pos > neg:   mapped = "joy"
                elif neg > pos: mapped = "anger"
                else:           mapped = "neutral"
            counter[mapped] += 1
        except:
            counter["neutral"] += 1
    return counter

def link_entity_emotions(entities, comments):
    out = {}
    for name, freq in entities[:5]:
        rel = [c for c in comments if name.lower() in c.lower()]
        if len(rel) < 3: continue
        out[name] = {
            "emotion_counts":  detect_emotions(rel, max_c=MAX_ENTITY_COMMENTS),
            "mention_count":   freq,
            "sample_comments": rel[:5],
        }
        top = out[name]["emotion_counts"].most_common(1)
        print(f"    {name:18s} | {freq:3d} mentions | "
              f"{top[0][0] if top else 'N/A'}")
    return out


# ════════════════════════════════════════════════════════════
# SECTION 9 — STANCE DETECTION
# ════════════════════════════════════════════════════════════

SUPPORT_KEYWORDS = {
    "support","agree","great","good","excellent","best","wonderful",
    "amazing","fantastic","love","like","appreciate","thank","congratulate",
    "well done","proud","happy","brilliant","superb","outstanding","perfect",
    "right","correct","true","indeed","absolutely","definitely","yes",
    "approve","favour","favor","endorse","back","promote","recommend",
    "built","developed","improved","progressed","achieved","succeeded",
    "delivered","accomplished","transformed","changed","growth","benefit",
    "helpful","useful","needed","necessary","important","positive",
    "hope","future","better","progress","development","opportunity",
}

OPPOSE_KEYWORDS = {
    "oppose","disagree","bad","wrong","terrible","worst","hate","dislike",
    "stupid","useless","waste","corrupt","fail","disaster","shame","awful",
    "horrible","pathetic","disgusting","ridiculous","nonsense","joke",
    "no","never","stop","enough","against","reject","refuse",
    "scam","fraud","liar","cheat","stolen","fake","propaganda","lie",
    "blame","fault","mistake","error","failed","inefficient",
    "expensive","costly","overpriced","burden","tax","unfair","unjust",
    "destroy","damage","harm","hurt","suffer","loss","problem","issue",
    "disappointed","frustrated","angry","upset","worried","concerned",
    "danger","risk","threat","crisis","poor","lack","missing","incomplete",
}

def detect_stance(comments, max_c=200):
    if not comments:
        return Counter({"Neutral": 1}), {}
    stance_counter = Counter()
    for c in comments[:max_c]:
        words   = set(c.lower().split())
        s_score = len(words & SUPPORT_KEYWORDS)
        o_score = len(words & OPPOSE_KEYWORDS)
        diff    = s_score - o_score
        if diff > 0:   stance_counter["Support"] += 1
        elif diff < 0: stance_counter["Oppose"]  += 1
        else:          stance_counter["Neutral"]  += 1
    total = sum(stance_counter.values()) or 1
    pct   = {k: round(v/total*100, 1) for k, v in stance_counter.items()}
    return stance_counter, pct

def link_entity_stance(entities, comments):
    out = {}
    for name, freq in entities[:5]:
        rel = [c for c in comments if name.lower() in c.lower()]
        if len(rel) < 3: continue
        counts, pct = detect_stance(rel, max_c=MAX_ENTITY_COMMENTS)
        dominant    = counts.most_common(1)[0][0] if counts else "Neutral"
        out[name]   = {
            "stance_counts":   counts,
            "stance_pct":      pct,
            "dominant_stance": dominant,
        }
        print(f"    {name:18s} | stance: {dominant} ({pct.get(dominant,0)}%)")
    return out


# ════════════════════════════════════════════════════════════
# SECTION 10 — ASPECT EXTRACTION
# ════════════════════════════════════════════════════════════

def get_cluster_keywords(comments, top_n=4):
    if not comments: return ["general"]
    try:
        vec    = TfidfVectorizer(max_features=100,
                                 stop_words="english", min_df=1)
        X      = vec.fit_transform(comments)
        scores = np.asarray(X.sum(axis=0)).flatten()
        vocab  = vec.get_feature_names_out()
        top    = np.argsort(scores)[-top_n:][::-1]
        kws    = [vocab[i] for i in top if len(vocab[i]) > 3]
        del X, vec; gc.collect()
        return kws if kws else ["general"]
    except Exception:
        return ["general"]

def textrank_summary(comments, n_sentences=2):
    sentences = []
    for c in comments[:15]:
        for sent in re.split(r"[.!?]", c):
            sent = sent.strip()
            if len(sent.split()) >= 5:
                sentences.append(sent)
    if len(sentences) < 2:
        return " ".join(comments[:2]) if comments else "No data."
    try:
        vec      = TfidfVectorizer(stop_words="english", max_features=150)
        tfidf    = vec.fit_transform(sentences)
        centroid = np.asarray(tfidf.mean(axis=0))
        scores   = cosine_similarity(tfidf, centroid).flatten()
        top_idx  = sorted(np.argsort(scores)[-n_sentences:])
        chosen   = [sentences[i] for i in top_idx]
        del tfidf, vec, scores; gc.collect()
        return " ".join(chosen)
    except Exception:
        return " ".join(comments[:2]) if comments else "No data."

def extract_aspects(entity_name, comments, n_aspects=4):
    if len(comments) < 10:
        return [{
            "label":    entity_name,
            "summary":  textrank_summary(comments),
            "count":    len(comments),
            "emotion":  detect_emotions(comments, max_c=50)
                        .most_common(1)[0][0] if comments else "neutral",
            "comments": comments[:5],
        }]
    n = min(n_aspects, len(comments) // 5, 5)
    if n < 2:
        return [{
            "label":    " ".join(get_cluster_keywords(comments)),
            "summary":  textrank_summary(comments),
            "count":    len(comments),
            "emotion":  detect_emotions(comments, max_c=50)
                        .most_common(1)[0][0] if comments else "neutral",
            "comments": comments[:5],
        }]
    try:
        vec  = TfidfVectorizer(max_features=200, stop_words="english",
                               min_df=1, ngram_range=(1,2))
        X    = vec.fit_transform(comments)
        lbls = KMeans(n_clusters=n, random_state=42,
                      n_init=5, max_iter=100).fit_predict(X)
        del X, vec; gc.collect()
    except Exception:
        return [{
            "label":    " ".join(get_cluster_keywords(comments)),
            "summary":  textrank_summary(comments),
            "count":    len(comments),
            "emotion":  "neutral",
            "comments": comments[:5],
        }]
    aspects = []
    for lbl in range(n):
        cl = [c for c, l in zip(comments, lbls) if l == lbl]
        if len(cl) < 3: continue
        kws     = get_cluster_keywords(cl, top_n=4)
        label   = " / ".join(kws[:3])
        emotion = detect_emotions(cl, max_c=30)
        dom_emo = emotion.most_common(1)[0][0] if emotion else "neutral"
        aspects.append({
            "label":    label,
            "summary":  textrank_summary(cl),
            "count":    len(cl),
            "emotion":  dom_emo,
            "comments": cl[:5],
        })
    aspects.sort(key=lambda x: x["count"], reverse=True)
    return aspects[:n_aspects]

def run_aspect_analysis(entities, comments):
    out = {}
    for name, freq in entities[:5]:
        rel = [c for c in comments if name.lower() in c.lower()]
        if len(rel) < 6: continue
        print(f"    Aspects for {name} ({len(rel)} comments)...")
        out[name] = extract_aspects(name, rel, n_aspects=4)
    return out


# ════════════════════════════════════════════════════════════
# SECTION 11 — OVERALL SUMMARIES (TextRank fallback)
# ════════════════════════════════════════════════════════════

def overall_textrank_fallback(topic, entities, clusters):
    reps = []
    for cl in list(clusters.values())[:6]:
        reps.extend(cl[:2])
    if not reps:
        return f"Public discussion around {topic}."
    entity_names = ", ".join([e[0] for e in entities[:5]])
    sentences    = []
    for c in [f"This is about {topic} involving {entity_names}."] + reps:
        for sent in re.split(r"[.!?]", c):
            sent = sent.strip()
            if len(sent.split()) >= 4:
                sentences.append(sent)
    if len(sentences) < 3:
        return f"Public discussion around {topic} involves {entity_names}."
    try:
        vec      = TfidfVectorizer(stop_words="english", max_features=200)
        tfidf    = vec.fit_transform(sentences)
        centroid = np.asarray(tfidf.mean(axis=0))
        scores   = cosine_similarity(tfidf, centroid).flatten()
        top_idx  = sorted(np.argsort(scores)[-4:])
        chosen   = [sentences[i] for i in top_idx if i > 0]
        del tfidf, vec, scores; gc.collect()
        return " ".join(chosen) if chosen else \
               f"Public discussion around {topic}."
    except Exception:
        return f"Public discussion around {topic} involves {entity_names}."


# ════════════════════════════════════════════════════════════
# SECTION 12 — CHARTS
# ════════════════════════════════════════════════════════════

def plot_dashboard(overall_emos, entities, entity_emos,
                   overall_stance, entity_stances, topic):
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle(f'Opinion Mining Dashboard — "{topic}"',
                 fontsize=15, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

    # Chart 1 — Overall emotion bar
    ax1 = fig.add_subplot(gs[0, 0])
    if overall_emos:
        labels = list(overall_emos.keys())
        values = list(overall_emos.values())
        colors = [EMO_COLORS.get(l,"#aaa") for l in labels]
        bars = ax1.bar(labels, values, color=colors,
                       edgecolor="white", linewidth=1, width=0.65)
        for b in bars:
            ax1.text(b.get_x()+b.get_width()/2,
                     b.get_height()+0.3,
                     str(int(b.get_height())),
                     ha="center", fontsize=8, fontweight="bold")
        ax1.set_title("Overall Emotion Distribution",
                      fontweight="bold", fontsize=11)
        ax1.set_xlabel("Emotion"); ax1.set_ylabel("Comments")
        ax1.set_facecolor("#F8F9FA")
        ax1.spines[["top","right"]].set_visible(False)

    # Chart 2 — Overall stance bar
    ax2 = fig.add_subplot(gs[0, 1])
    if overall_stance:
        s_labels = list(overall_stance.keys())
        s_values = list(overall_stance.values())
        s_colors = [STANCE_COLORS.get(l,"#aaa") for l in s_labels]
        bars = ax2.bar(s_labels, s_values, color=s_colors,
                       edgecolor="white", linewidth=1, width=0.5)
        for b in bars:
            ax2.text(b.get_x()+b.get_width()/2,
                     b.get_height()+0.3,
                     str(int(b.get_height())),
                     ha="center", fontsize=9, fontweight="bold")
        ax2.set_title("Overall Stance Distribution",
                      fontweight="bold", fontsize=11)
        ax2.set_xlabel("Stance"); ax2.set_ylabel("Comments")
        ax2.set_facecolor("#F8F9FA")
        ax2.spines[["top","right"]].set_visible(False)

    # Chart 3 — Entity mention frequency
    ax3 = fig.add_subplot(gs[0, 2])
    if entities:
        names  = [e[0] for e in entities[:8]]
        counts = [e[1] for e in entities[:8]]
        clrs   = plt.cm.Blues(np.linspace(0.4,0.9,len(names)))[::-1]
        bars = ax3.barh(names[::-1], counts[::-1],
                        color=clrs[::-1], edgecolor="white")
        for b in bars:
            ax3.text(b.get_width()+0.2,
                     b.get_y()+b.get_height()/2,
                     str(int(b.get_width())),
                     va="center", fontsize=8, fontweight="bold")
        ax3.set_title("Top Mentioned Entities",
                      fontweight="bold", fontsize=11)
        ax3.set_xlabel("Mentions")
        ax3.set_facecolor("#F8F9FA")
        ax3.spines[["top","right"]].set_visible(False)

    # Chart 4 — Entity × Emotion grouped bar
    ax4 = fig.add_subplot(gs[1, 0])
    if entity_emos:
        ent_names = list(entity_emos.keys())[:5]
        n   = len(ALL_EMOTIONS)
        x   = np.arange(len(ent_names))
        w   = 0.11
        for j, emo in enumerate(ALL_EMOTIONS):
            vals = [entity_emos[e]["emotion_counts"].get(emo,0)
                    for e in ent_names]
            ax4.bar(x+(j-n/2)*w, vals, w,
                    label=emo.capitalize(),
                    color=EMO_COLORS[emo],
                    edgecolor="white", linewidth=0.3)
        ax4.set_xticks(x)
        ax4.set_xticklabels(ent_names, fontsize=8, fontweight="bold")
        ax4.set_title("Entity-wise Emotion  (Core Novelty)",
                      fontweight="bold", fontsize=11)
        ax4.set_xlabel("Entities"); ax4.set_ylabel("Comments")
        ax4.legend(title="Emotion", bbox_to_anchor=(1.01,1),
                   loc="upper left", fontsize=7)
        ax4.set_facecolor("#F8F9FA")
        ax4.spines[["top","right"]].set_visible(False)

    # Chart 5 — Entity × Stance grouped bar
    ax5 = fig.add_subplot(gs[1, 1])
    if entity_stances:
        ent_names   = list(entity_stances.keys())[:5]
        stance_lbls = ["Support","Oppose","Neutral"]
        x = np.arange(len(ent_names))
        w = 0.22
        for j, st in enumerate(stance_lbls):
            vals = [entity_stances[e]["stance_counts"].get(st,0)
                    for e in ent_names]
            ax5.bar(x+(j-1)*w, vals, w,
                    label=st, color=STANCE_COLORS[st],
                    edgecolor="white", linewidth=0.5)
        ax5.set_xticks(x)
        ax5.set_xticklabels(ent_names, fontsize=8, fontweight="bold")
        ax5.set_title("Entity-wise Stance",
                      fontweight="bold", fontsize=11)
        ax5.set_xlabel("Entities"); ax5.set_ylabel("Comments")
        ax5.legend(fontsize=8)
        ax5.set_facecolor("#F8F9FA")
        ax5.spines[["top","right"]].set_visible(False)

    # Chart 6 — Legend panel
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.axis("off")
    ax6.text(0.5, 0.95, "What This Dashboard Shows",
             ha="center", va="top", fontsize=11,
             fontweight="bold", color="#0D1B3E",
             transform=ax6.transAxes)
    info = [
        "① Emotion  — How people FEEL",
        "   (joy, anger, sadness, etc.)",
        "",
        "② Stance   — What people THINK",
        "   (support / oppose / neutral)",
        "",
        "③ Entities — Who is discussed",
        "   (people, orgs, places)",
        "",
        "④ Aspects  — Sub-topics debated",
        "   per entity (see cards below)",
        "",
        "⑤ AI Image — Gemini infographic",
        "   (see below dashboard)",
    ]
    y = 0.82
    for line in info:
        color = "#0E6BA8" if line.startswith(
            ("①","②","③","④","⑤")) else "#64748B"
        ax6.text(0.08, y, line, ha="left", va="top",
                 fontsize=9.5, color=color,
                 transform=ax6.transAxes)
        y -= 0.08

    fig.patch.set_facecolor("white")
    path = f"{OUTPUT_DIR}/{safe(topic)}_dashboard.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig); gc.collect()
    print(f"  Saved → {path}")
    return path


# ════════════════════════════════════════════════════════════
# SECTION 13 — DISPLAY HELPERS
# ════════════════════════════════════════════════════════════

def section_header(title):
    display(HTML(
        f'<div style="background:#0D1B3E;padding:9px 16px;'
        f'border-radius:7px;margin:16px 0 8px 0;">'
        f'<span style="color:#E9C46A;font-size:15px;font-weight:bold;">'
        f'{title}</span></div>'
    ))

def progress(step, total, msg):
    pct = int(step/total*100)
    bar = "█"*(pct//5) + "░"*(20-pct//5)
    print(f"  [{bar}] {pct:3d}%  {msg}", flush=True)

def timer_badge(seconds):
    display(HTML(
        f'<div style="display:inline-block;background:#166534;'
        f'color:white;padding:4px 14px;border-radius:20px;'
        f'font-size:12px;font-weight:bold;margin:6px 0;">'
        f'⚡ Completed in {seconds:.1f} seconds</div>'
    ))

def stance_badge(pct, dominant):
    return "".join([
        f'<span style="background:{STANCE_COLORS.get(k,"#aaa")};'
        f'color:white;padding:2px 10px;border-radius:10px;font-size:11px;'
        f'margin-right:5px;font-weight:bold;">{k} {v}%</span>'
        for k, v in pct.items()
    ])

def aspect_block(aspects):
    html = '<div style="margin-top:12px;">'
    html += ('<div style="font-size:12px;font-weight:bold;'
             'color:#0D1B3E;margin-bottom:6px;">'
             '🔍 Aspect-Based Analysis (Gemini AI):</div>')
    for asp in aspects:
        ec = EMO_COLORS.get(asp["emotion"], "#64748B")
        html += (
            f'<div style="background:#F8FAFC;border:1px solid #E2E8F0;'
            f'border-left:4px solid {ec};border-radius:6px;'
            f'padding:8px 12px;margin-bottom:6px;">'
            f'<div style="display:flex;justify-content:space-between;'
            f'align-items:center;margin-bottom:4px;">'
            f'<span style="font-size:11px;font-weight:bold;color:#0D1B3E;'
            f'text-transform:capitalize;">📌 {asp["label"]}</span>'
            f'<span style="font-size:10px;background:{ec};color:white;'
            f'padding:1px 8px;border-radius:8px;">'
            f'{asp["emotion"]} · {asp["count"]} comments</span></div>'
            f'<div style="font-size:12px;color:#475569;line-height:1.6;">'
            f'{asp["summary"]}</div></div>'
        )
    html += '</div>'
    return html

def entity_card_full(name, mention_count, emotion_counts,
                     stance_data, aspects, summary_text):
    top3     = emotion_counts.most_common(3)
    emo_html = "".join([
        f'<span style="background:{EMO_COLORS.get(e,"#aaa")};'
        f'color:white;padding:2px 10px;border-radius:10px;'
        f'font-size:11px;margin-right:4px;font-weight:bold;">'
        f'{e.capitalize()} {c}</span>'
        for e, c in top3
    ])
    stance_html = ""
    if stance_data:
        pct = stance_data.get("stance_pct", {})
        dom = stance_data.get("dominant_stance", "Neutral")
        stance_html = (
            f'<div style="margin:8px 0 4px 0;">'
            f'<span style="font-size:11px;color:#64748B;margin-right:6px;">'
            f'Stance:</span>' + stance_badge(pct, dom) + '</div>'
        )
    asp_html = aspect_block(aspects) if aspects else ""
    display(HTML(
        f'<div style="border:1px solid #D1DBE8;border-radius:10px;'
        f'padding:16px 20px;margin:8px 0;background:white;">'
        f'<div style="display:flex;justify-content:space-between;'
        f'align-items:center;margin-bottom:10px;">'
        f'<span style="font-size:16px;font-weight:bold;color:#0D1B3E;">'
        f'🏷️ {name}</span>'
        f'<span style="font-size:11px;color:#64748B;background:#F4F7FB;'
        f'padding:2px 8px;border-radius:8px;">{mention_count} mentions</span>'
        f'</div>'
        f'<div style="margin-bottom:6px;">'
        f'<span style="font-size:11px;color:#64748B;margin-right:6px;">'
        f'Emotion:</span>{emo_html}</div>'
        + stance_html +
        f'<div style="font-size:13px;color:#1E293B;line-height:1.7;'
        f'border-left:3px solid #0A9396;padding:8px 12px;'
        f'background:#F4F7FB;border-radius:0 5px 5px 0;margin-top:10px;">'
        f'🤖 {summary_text}</div>'
        + asp_html +
        f'</div>'
    ))

def summary_box(text, title="📝 AI Summary", color="#0D1B3E"):
    display(HTML(
        f'<div style="background:{color};border-radius:9px;'
        f'padding:16px 20px;margin:8px 0;">'
        f'<div style="color:#E9C46A;font-weight:bold;font-size:14px;'
        f'margin-bottom:8px;">{title}</div>'
        f'<div style="color:white;font-size:13.5px;line-height:1.8;">'
        f'{text}</div></div>'
    ))


# ── GEMINI TEXT SUMMARIES ─────────────────────────────────────
import google.generativeai as genai
import time as _time

genai.configure(api_key=GEMINI_API_KEY)
gemini_text = genai.GenerativeModel("gemini-1.5-flash-8b")

def gemini_call(prompt, max_tokens=120):
    try:
        response = gemini_text.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=max_tokens,
                temperature=0.3,
            )
        )
        _time.sleep(2)
        return response.text.strip()
    except Exception as e:
        print(f"    ⚠ Gemini text error: {e}")
        _time.sleep(5)
        return None

def gemini_overall_summary(topic, entities, overall_emos, stance_pct):
    entity_str = ", ".join([e[0] for e in entities[:5]])
    top_emo    = overall_emos.most_common(1)[0][0] if overall_emos else "neutral"
    sup = stance_pct.get("Support", 0)
    opp = stance_pct.get("Oppose",  0)
    neu = stance_pct.get("Neutral", 0)
    prompt = (
        f"News topic: {topic}\n"
        f"Key entities: {entity_str}\n"
        f"Dominant emotion: {top_emo}\n"
        f"Stance: {sup}% Support, {opp}% Oppose, {neu}% Neutral\n\n"
        f"Write a 3-sentence summary of overall public opinion on this topic."
    )
    result = gemini_call(prompt, max_tokens=150)
    if result: print(f"    ✓ Overall summary by Gemini")
    return result

def gemini_stance_summary(topic, stance_counts, stance_pct):
    dominant = stance_counts.most_common(1)[0][0] if stance_counts else "Neutral"
    sup = stance_pct.get("Support", 0)
    opp = stance_pct.get("Oppose",  0)
    neu = stance_pct.get("Neutral", 0)
    prompt = (
        f"Topic: {topic}\n"
        f"Stance: {sup}% Support, {opp}% Oppose, {neu}% Neutral\n"
        f"Dominant: {dominant}\n\n"
        f"Write 2 sentences analysing what this stance distribution "
        f"tells us about public opinion on this topic."
    )
    return gemini_call(prompt, max_tokens=100)

def gemini_entity_summary(entity_name, comments, emotion_counts, stance_pct):
    top_emos = emotion_counts.most_common(3)
    dominant = top_emos[0][0] if top_emos else "neutral"
    emo_str  = ", ".join([f"{e}({c})" for e, c in top_emos])
    sup      = stance_pct.get("Support", 0)
    opp      = stance_pct.get("Oppose",  0)
    comment_str = " | ".join(comments[:5])
    prompt = (
        f"Entity: {entity_name}\n"
        f"Dominant emotion: {dominant}\n"
        f"Emotions: {emo_str}\n"
        f"Stance: {sup}% Support, {opp}% Oppose\n"
        f"Comments: {comment_str}\n\n"
        f"Write 2-3 sentences summarising how the public feels "
        f"about {entity_name}."
    )
    result = gemini_call(prompt, max_tokens=120)
    if result: print(f"    ✓ {entity_name} summary by Gemini")
    return result

def gemini_aspect_summary(entity_name, aspect_label, comments, emotion):
    comment_str = " | ".join(comments[:5])
    prompt = (
        f"Entity: {entity_name}\n"
        f"Aspect: {aspect_label}\n"
        f"Dominant emotion: {emotion}\n"
        f"Comments: {comment_str}\n\n"
        f"Write exactly 2 sentences summarising what people "
        f"are saying about this aspect."
    )
    return gemini_call(prompt, max_tokens=90)

# ════════════════════════════════════════════════════════════
# SECTION 14 — MAIN PIPELINE
# ════════════════════════════════════════════════════════════

def run(topic):
    t_start = time.time()
    clear_output(wait=True)
    display(HTML(
        f'<div style="background:#0D1B3E;padding:14px 22px;'
        f'border-radius:10px;margin-bottom:14px;">'
        f'<div style="color:#E9C46A;font-size:19px;font-weight:bold;">'
        f'🔍 Analysing: &quot;{topic}&quot;</div>'
        f'<div style="color:#D9E8F5;font-size:12px;margin-top:4px;">'
        f'Emotion · Stance · Aspect · Gemini Image  |  NIT Warangal'
        f'</div></div>'
    ))

    STEPS = 11

    # 1. Search
    progress(1, STEPS, "Searching YouTube...")
    video_ids = set()
    for q in [topic, topic + " news"]:
        video_ids.update(search_videos(q, max_results=15))
    print(f"    {len(video_ids)} videos found")

    # 2. Collect
    progress(2, STEPS, f"Collecting comments (max {MAX_RAW_COMMENTS})...")
    raw_comments = []
    video_count  = 0
    for vid in video_ids:
        if len(raw_comments) >= MAX_RAW_COMMENTS: break
        if video_count >= MAX_VIDEOS: break
        det = video_details(vid)
        if not det: continue
        title_v, channel, views, n_comments = det
        if views < MIN_VIEWS: continue
        if n_comments < 100: continue
        print(f"    ✓ {title_v[:50]}  ({views:,} views)")
        raw_comments.extend(get_comments_fast(vid, max_comments=300))
        video_count += 1
    print(f"    {len(raw_comments)} raw comments")

    if not raw_comments:
        display(HTML('<p style="color:red;font-weight:bold;">'
                     '❌ No comments found. Try a different topic.</p>'))
        return

    # 3. Preprocess
    progress(3, STEPS, "Preprocessing...")
    cleaned  = [clean(c) for c in raw_comments]
    del raw_comments; gc.collect()
    filtered = filter_comments(cleaned)
    del cleaned; gc.collect()
    unique   = list(set(filtered))
    del filtered; gc.collect()
    sample   = random.sample(unique, min(MAX_SAMPLE, len(unique)))
    del unique; gc.collect()
    print(f"    {len(sample)} comments ready")
    mem()

    # 4. NER
    progress(4, STEPS, "Extracting named entities...")
    entities = extract_entities(sample)
    print(f"    Entities: {[e[0] for e in entities]}")

    # 5. Cluster
    progress(5, STEPS, "Clustering comments...")
    clusters = cluster_comments(sample)

    # 6. Emotion
    progress(6, STEPS, "Detecting emotions (NRCLex)...")
    overall_emos = detect_emotions(sample, max_c=MAX_EMOTION_COMMENTS)
    entity_emos  = link_entity_emotions(entities, sample)
    print(f"    Overall: {dict(overall_emos)}")

    # 7. Stance
    progress(7, STEPS, "Detecting stance...")
    overall_stance, overall_stance_pct = detect_stance(
        sample, max_c=MAX_EMOTION_COMMENTS)
    entity_stances = link_entity_stance(entities, sample)
    print(f"    Stance: {dict(overall_stance)}")

    # 8. Aspects
    progress(8, STEPS, "Extracting aspects per entity...")
    entity_aspects = run_aspect_analysis(entities, sample)

    # 9. Summaries (Gemini text + TextRank fallback)
    progress(9, STEPS, "Generating summaries (Gemini AI)...")

    ov_summary = gemini_overall_summary(
        topic, entities, overall_emos, overall_stance_pct
    ) or overall_textrank_fallback(topic, entities, clusters)

    stance_summ = gemini_stance_summary(
        topic, overall_stance, overall_stance_pct
    ) or (f"Stance: {overall_stance_pct.get('Support',0)}% Support, "
          f"{overall_stance_pct.get('Oppose',0)}% Oppose, "
          f"{overall_stance_pct.get('Neutral',0)}% Neutral.")

    entity_summs = {}
    for name, edata in entity_emos.items():
        st_pct = entity_stances.get(name, {}).get("stance_pct", {})
        result = gemini_entity_summary(
            name, edata["sample_comments"],
            edata["emotion_counts"], st_pct)
        entity_summs[name] = result or textrank_summary(
            edata["sample_comments"])

    for name, aspects in entity_aspects.items():
        for asp in aspects:
            result = gemini_aspect_summary(
                name, asp["label"],
                asp.get("comments", []), asp["emotion"])
            if result:
                asp["summary"] = result
    mem()

    # 10. Charts
    progress(10, STEPS, "Rendering dashboard...")
    section_header("📊 Opinion Mining Dashboard")
    plot_dashboard(overall_emos, entities, entity_emos,
                   overall_stance, entity_stances, topic)

    # 11. Gemini AI Image  ★ NEW ★
    progress(11, STEPS, "Generating AI analytics image (Gemini)...")
    section_header("🎨 AI-Generated Analytics Image")
    generate_analytics_image(
        topic            = topic,
        entities         = entities,
        overall_emos     = overall_emos,
        overall_stance_pct = overall_stance_pct,
        entity_emos      = entity_emos,
    )

    # Text outputs
    section_header("📝 Overall Summary  (Gemini AI)")
    summary_box(ov_summary, "📝 Overall Summary")

    section_header("⚖️ Stance Analysis  (Gemini AI)")
    summary_box(stance_summ, "⚖️ Stance Summary", "#1e3a5f")

    section_header("🏷️ Entity Analysis  (Emotion + Stance + Aspects)")
    for name, edata in entity_emos.items():
        entity_card_full(
            name           = name,
            mention_count  = edata["mention_count"],
            emotion_counts = edata["emotion_counts"],
            stance_data    = entity_stances.get(name),
            aspects        = entity_aspects.get(name, []),
            summary_text   = entity_summs.get(name, "—"),
        )

    total = time.time() - t_start
    timer_badge(total)
    mem()

    # Save report
    rpt = f"{OUTPUT_DIR}/{safe(topic)}_report.txt"
    with open(rpt, "w") as f:
        f.write(f"TOPIC: {topic}\nTime: {total:.1f}s\n{'='*55}\n\n")
        f.write("ENTITIES:\n")
        for n, c in entities: f.write(f"  {n}: {c}\n")
        f.write("\nOVERALL EMOTIONS:\n")
        for e, c in overall_emos.most_common():
            f.write(f"  {e}: {c}\n")
        f.write("\nOVERALL STANCE:\n")
        for s, p in overall_stance_pct.items():
            f.write(f"  {s}: {p}%\n")
        f.write(f"\nOVERALL SUMMARY:\n{ov_summary}\n")
        f.write(f"\nSTANCE SUMMARY:\n{stance_summ}\n")
        f.write("\nENTITY ANALYSIS:\n")
        for name in entity_emos:
            f.write(f"\n  [{name}]\n")
            f.write(f"  Summary: {entity_summs.get(name,'')}\n")
            st = entity_stances.get(name, {})
            f.write(f"  Stance: {dict(st.get('stance_pct',{}))}\n")
            f.write("  Aspects:\n")
            for asp in entity_aspects.get(name, []):
                f.write(f"    - {asp['label']} ({asp['count']} comments)\n")
                f.write(f"      {asp['summary']}\n")

    display(HTML(
        f'<div style="background:#14532D;color:white;padding:10px 18px;'
        f'border-radius:8px;margin-top:12px;font-size:12px;">'
        f'✅ Done! Report + Dashboard + AI Image → <b>{OUTPUT_DIR}/</b>'
        f'</div>'
    ))
    gc.collect()


# ════════════════════════════════════════════════════════════
# SECTION 15 — WIDGET
# ════════════════════════════════════════════════════════════

display(HTML("""
<div style="background:#0D1B3E;padding:20px 24px;border-radius:11px;
            margin-bottom:10px;">
  <div style="color:#E9C46A;font-size:20px;font-weight:bold;
              margin-bottom:6px;">
    🤖 Social Media Opinion Mining Agent
    <span style="font-size:12px;background:#7c3aed;
    padding:2px 10px;border-radius:10px;vertical-align:middle;">
    ✨ Gemini AI</span>
  </div>
  <div style="color:#D9E8F5;font-size:13px;line-height:1.8;">
    Enter a news headline → click <b>Analyse</b><br>
    &nbsp;&nbsp;① Emotions — NRCLex lexicon<br>
    &nbsp;&nbsp;② Stance — Support / Oppose / Neutral<br>
    &nbsp;&nbsp;③ Aspects — Sub-topics per entity<br>
    &nbsp;&nbsp;④ Summaries — Gemini 1.5 Flash<br>
    &nbsp;&nbsp;⑤ Analytics Image — Gemini Image Generation
  </div>
</div>"""))

topic_box = widgets.Text(
    placeholder="e.g. Ganga Expressway / Budget 2025 / India Pakistan",
    layout=widgets.Layout(width="74%", height="36px"),
    style={"description_width": "0px"}
)
run_btn = widgets.Button(
    description=" Analyse", icon="bolt",
    layout=widgets.Layout(width="138px", height="36px"),
    style={"button_color": "#0E6BA8", "font_weight": "bold"}
)
out = widgets.Output()

def on_click(b):
    topic = topic_box.value.strip()
    if not topic:
        with out:
            display(HTML('<p style="color:#E63946;font-weight:bold;">'
                         '⚠ Please enter a topic.</p>'))
        return
    with out:
        clear_output(wait=True)
        run(topic)

run_btn.on_click(on_click)
display(widgets.HBox([topic_box, run_btn],
        layout=widgets.Layout(margin="5px 0 8px 0")))
display(out)